# グルモジ — Google Colab版

上から順にセルを実行すると、話者分離文字起こしWeb UIをColab内に表示します。

1. Colabの「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU以上** を選択してください。
2. Hugging Faceで `pyannote/speaker-diarization-3.1`、`pyannote/segmentation-3.0`、`pyannote/speaker-diarization-community-1` の利用条件へ同意してください。
3. トークン入力は非表示で行われ、ノートブックの出力には表示されません。

> ColabのランタイムとGPUは保証されず、停止すると一時ファイルは消えます。出力を保持する場合はGoogle Drive保存を有効にしてください。Web UIは文字起こしという対話的計算のためだけに使用してください。

In [ ]:
# 1. リポジトリと依存関係を準備（初回は数分かかります）
from pathlib import Path
import hashlib
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError("このノートブックはGoogle Colabで実行してください。") from exc

REPO_URL = "https://github.com/krokharu/gurumoji.git"
REPO_DIR = Path("/content/gurumoji")
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "fonts-noto-cjk"], check=True)

requirements = (REPO_DIR / "requirements.txt").read_bytes()
setup_hash = hashlib.sha256(requirements + b"colab-cu128-v1").hexdigest()[:12]
setup_marker = Path(f"/content/.gurumoji_setup_{setup_hash}")
if not setup_marker.exists():
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "torch==2.8.0+cu128", "torchvision==0.23.0+cu128", "torchaudio==2.8.0+cu128",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "-r", str(REPO_DIR / "requirements.txt"),
    ], check=True)
    subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
    setup_marker.touch()

os.chdir(REPO_DIR)
print("準備完了:", REPO_DIR)


In [ ]:
# 2. APIトークンを安全に設定
from getpass import getpass
import json

token_path = REPO_DIR / "tokens.json"
tokens = {}
if token_path.is_file():
    try:
        tokens = json.loads(token_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        tokens = {}

def secret_value(key, prompt, required=False):
    current = str(tokens.get(key, "") or "").strip()
    suffix = "（Enterで現在値を維持）" if current else ""
    value = getpass(prompt + suffix + ": ").strip()
    result = value or current
    if required and not result:
        raise ValueError(f"{key} は話者分離に必須です。")
    return result

tokens.update({
    "huggingface_token": secret_value("huggingface_token", "Hugging Face read token", required=True),
    "openai_api_key": secret_value("openai_api_key", "OpenAI API key（任意）"),
    "google_api_key": secret_value("google_api_key", "Google Gemini API key（任意）"),
    "openai_model": tokens.get("openai_model") or "gpt-5.6-luna",
    "google_model": tokens.get("google_model") or "gemini-3.5-flash",
})
token_path.write_text(json.dumps(tokens, ensure_ascii=False, indent=2), encoding="utf-8")
print("トークン設定を保存しました（値は表示しません）。")


In [ ]:
# 3. 出力・話者台帳をGoogle Driveへ保持するか選択
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    storage_root = Path("/content/drive/MyDrive/gurumoji")
else:
    storage_root = REPO_DIR

output_dir = storage_root / "output"
data_dir = storage_root / "data"
output_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)
os.environ["MOJIOKOSI_OUTPUT_DIR"] = str(output_dir)
os.environ["MOJIOKOSI_DATA_DIR"] = str(data_dir)
print("出力:", output_dir)
print("話者台帳・ライブラリ:", data_dir)


In [ ]:
# 4. AIST感情分析を使う場合だけ True（任意・追加ダウンロードあり）
ENABLE_AIST_EMOTION = False

if ENABLE_AIST_EMOTION:
    s3prl_root = REPO_DIR / "models" / "s3prl-v0.4.17"
    if not (s3prl_root / "s3prl" / "run_downstream.py").is_file():
        s3prl_root.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run([
            "git", "clone", "--branch", "v0.4.17", "--depth", "1",
            "https://github.com/s3prl/s3prl.git", str(s3prl_root),
        ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "-r", str(REPO_DIR / "requirements-emotion.txt"),
    ], check=True)
    os.environ["MOJIOKOSI_S3PRL_ROOT"] = str(s3prl_root)
    print("AIST感情分析の実行環境を準備しました。")
else:
    print("AIST感情分析の追加セットアップは省略します。")


In [ ]:
# 5. Web UIを起動してColab内に表示
from google.colab import output
import time
import urllib.request

PORT = 7860
if "gurumoji_server" in globals() and gurumoji_server.poll() is None:
    gurumoji_server.terminate()
    gurumoji_server.wait(timeout=15)

server_log_path = Path("/content/gurumoji_server.log")
server_log = server_log_path.open("w", encoding="utf-8")
server_env = os.environ.copy()
server_env.update({
    "MOJIOKOSI_RUNTIME": "colab",
    "MOJIOKOSI_NO_BROWSER": "1",
    "MOJIOKOSI_HOST": "127.0.0.1",
    "MOJIOKOSI_PORT": str(PORT),
})
gurumoji_server = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd=str(REPO_DIR),
    env=server_env,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

for _ in range(90):
    if gurumoji_server.poll() is not None:
        server_log.flush()
        raise RuntimeError(server_log_path.read_text(encoding="utf-8", errors="replace")[-5000:])
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise TimeoutError("Web UIの起動がタイムアウトしました。/content/gurumoji_server.log を確認してください。")

print("グルモジを起動しました。下の画面からファイルをアップロードできます。")
output.serve_kernel_port_as_iframe(PORT, height=1100)


In [ ]:
# 6. 終了するときだけ実行
if "gurumoji_server" in globals() and gurumoji_server.poll() is None:
    gurumoji_server.terminate()
    gurumoji_server.wait(timeout=15)
    print("グルモジを停止しました。")
else:
    print("グルモジは停止済みです。")
